In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [54]:
df = pd.read_csv('/Users/amodi/Documents/DTSC2301PJ2/Dataset/matches.csv') 

In [55]:
df.info 

<bound method DataFrame.info of            id   season        city        date   match_type  player_of_match  \
0      335982  2007/08   Bangalore  2008-04-18       League      BB McCullum   
1      335983  2007/08  Chandigarh  2008-04-19       League       MEK Hussey   
2      335984  2007/08       Delhi  2008-04-19       League      MF Maharoof   
3      335985  2007/08      Mumbai  2008-04-20       League       MV Boucher   
4      335986  2007/08     Kolkata  2008-04-20       League        DJ Hussey   
...       ...      ...         ...         ...          ...              ...   
1090  1426307     2024   Hyderabad  2024-05-19       League  Abhishek Sharma   
1091  1426309     2024   Ahmedabad  2024-05-21  Qualifier 1         MA Starc   
1092  1426310     2024   Ahmedabad  2024-05-22   Eliminator         R Ashwin   
1093  1426311     2024     Chennai  2024-05-24  Qualifier 2    Shahbaz Ahmed   
1094  1426312     2024     Chennai  2024-05-26        Final         MA Starc   

       

In [56]:
df.shape

(1095, 20)

In [57]:
df.describe().T 

,count,mean,std,min,25%,50%,75%,max
id,1095.0,904828.319635,367740.242299,335982.0,548331.5,980961.0,1254062.5,1426312.0
result_margin,1076.0,17.259294,21.787444,1.0,6.0,8.0,20.0,146.0
target_runs,1092.0,165.684066,33.427048,43.0,146.0,166.0,187.0,288.0
target_overs,1092.0,19.759341,1.581108,5.0,20.0,20.0,20.0,20.0


In [58]:
df.isnull().sum()

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

In [59]:
df.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [60]:
#Convert date from text to real dates, and create a clean year column
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# renamed franchises
team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    df[col] = df[col].replace(team_map)

#Drop matches with no result
df = df[df['winner'].notna()].copy()

#Put matches in time order
df = df.sort_values('date').reset_index(drop=True)

In [61]:
print(df.shape)
print(df['date'].dtype)
df[['season', 'year', 'date', 'team1', 'team2', 'winner']].head() 

(1090, 21)
datetime64[ns]


,season,year,date,team1,team2,winner
0,2007/08,2008,2008-04-18,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders
1,2007/08,2008,2008-04-19,Punjab Kings,Chennai Super Kings,Chennai Super Kings
2,2007/08,2008,2008-04-19,Delhi Capitals,Rajasthan Royals,Delhi Capitals
3,2007/08,2008,2008-04-20,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru
4,2007/08,2008,2008-04-20,Kolkata Knight Riders,Deccan Chargers,Kolkata Knight Riders


In [62]:
h2h_records_team1 = []
h2h_records_team2 = []

for row in df.iterrows():
    current_team = row[1]['team1']
    opponent_team = row[1]['team2']
    date = row[1]['date']
    h2_history = df[((df['team1'] == current_team) & (df['team2'] == opponent_team)) | ((df['team1'] == opponent_team) & (df['team2'] == current_team))].loc[df['date'] < date]
    h2h_records_where_team1_won = h2_history[h2_history['winner'] == current_team]
    h2h_records_where_team2_won = h2_history[h2_history['winner'] == opponent_team]
    h2h_records_team1.append(len(h2h_records_where_team1_won))
    h2h_records_team2.append(len(h2h_records_where_team2_won))

h2h_records_team1
h2h_records_team2

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 2,
 1,
 2,
 2,
 2,
 1,
 1,
 0,
 2,
 1,
 2,
 1,
 2,
 1,
 1,
 2,
 2,
 0,
 2,
 2,
 3,
 3,
 3,
 2,
 2,
 1,
 2,
 2,
 1,
 1,
 1,
 2,
 0,
 1,
 2,
 3,
 1,
 1,
 3,
 2,
 0,
 3,
 1,
 2,
 2,
 2,
 2,
 3,
 3,
 0,
 3,
 3,
 0,
 1,
 1,
 0,
 2,
 1,
 2,
 1,
 2,
 2,
 2,
 2,
 3,
 2,
 1,
 3,
 2,
 4,
 1,
 2,
 4,
 0,
 0,
 2,
 2,
 3,
 2,
 1,
 2,
 2,
 1,
 1,
 3,
 4,
 3,
 2,
 2,
 4,
 3,
 2,
 1,
 4,
 3,
 2,
 3,
 2,
 2,
 2,
 3,
 3,
 2,
 3,
 3,
 1,
 2,
 2,
 1,
 2,
 4,
 2,
 5,
 5,
 3,
 4,
 4,
 3,
 1,
 5,
 0,
 3,
 0,
 2,
 4,
 4,
 6,
 0,
 4,
 2,
 0,
 4,
 4,
 0,
 4,
 0,
 3,
 0,
 0,
 4,
 4,
 3,
 4,
 3,
 0,
 0,
 1,
 1,
 0,
 2,
 3,
 0,
 0,
 3,
 0,
 4,
 4,
 0,
 5,
 4,
 1,
 4,
 0,
 3,
 5,
 3,
 0,
 1,
 4,
 0,
 3,
 3,
 3,
 0,
 3,
 3,
 3,
 1,
 0,
 4,
 1,
 0,
 3,
 2,
 4,
 5,
 4,
 1,
 5,
 5,
 4,
 3,
 0,


In [71]:
df['team1_h2h_wins'] = h2h_records_team1
df['team2_h2h_wins'] = h2h_records_team2
df['h2h_meetings'] = df['team1_h2h_wins'] + df['team2_h2h_wins']
df["h2h_team1_win_percentage"] = df["team1_h2h_wins"] / df["h2h_meetings"] * 100
df["h2h_team2_win_percentage"] = df["team2_h2h_wins"] / df["h2h_meetings"] * 100
    
df[['team1', 'team2', 'team1_h2h_wins', 'team2_h2h_wins', 'h2h_meetings', "h2h_team1_win_percentage", "h2h_team2_win_percentage"]]

,team1,team2,team1_h2h_wins,team2_h2h_wins,h2h_meetings,h2h_team1_win_percentage,h2h_team2_win_percentage
0,Royal Challengers Bengaluru,Kolkata Knight Riders,0,0,0,NaN,NaN
1,Punjab Kings,Chennai Super Kings,0,0,0,NaN,NaN
2,Delhi Capitals,Rajasthan Royals,0,0,0,NaN,NaN
3,Mumbai Indians,Royal Challengers Bengaluru,0,0,0,NaN,NaN
4,Kolkata Knight Riders,Deccan Chargers,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...
1085,Punjab Kings,Sunrisers Hyderabad,7,15,22,31.818182,68.181818
1086,Sunrisers Hyderabad,Kolkata Knight Riders,9,17,26,34.615385,65.384615
1087,Royal Challengers Bengaluru,Rajasthan Royals,15,13,28,53.571429,46.428571
1088,Sunrisers Hyderabad,Rajasthan Royals,10,9,19,52.631579,47.368421
